# Unidad 4 · Colab 2 de 3
## Desarrollo de APIs RESTful con FastAPI/Flask y Pydantic

**Objetivos de este notebook**

- Crear endpoints RESTful con **FastAPI**.
- Definir modelos de datos con **Pydantic** para validación automática y serialización.
- Entender la documentación interactiva automática (Swagger UI / OpenAPI).
- Comparar brevemente con **Flask**, la alternativa más liviana y menos opinionada.

> **Nivel:** intermedio. Se asume que ya consumiste APIs (Colab 1) y sabés Python con type hints básicos.

**Cómo probamos la API en este notebook:** usamos `TestClient` de FastAPI, que simula requests HTTP sin necesitar levantar un servidor real — ideal para Colab. Al final te mostramos cómo correrla de verdad con `uvicorn`.

---

## 1. FastAPI vs. Flask

| | FastAPI | Flask |
|---|---|---|
| Validación de datos | Automática, vía Pydantic | Manual (o con extensiones como flask-pydantic) |
| Documentación | Automática (Swagger UI / ReDoc) | Manual o con extensiones |
| Async nativo | Sí | Limitado (mejora en versiones recientes) |
| Curva de aprendizaje | Un poco más de magia (type hints) | Más minimalista y explícito |
| Cuándo usarlo | APIs nuevas, con muchos datos estructurados | Proyectos simples, o con mucho código legado |

Documentación oficial: [FastAPI](https://fastapi.tiangolo.com/) · [Flask](https://flask.palletsprojects.com/)

## 2. Primeros endpoints

```bash
pip install fastapi uvicorn
```

```python
from fastapi import FastAPI

app = FastAPI()

@app.get('/')
def inicio():
    return {'mensaje': 'API de productos'}

@app.get('/productos/{producto_id}')
def obtener_producto(producto_id: int):
    return {'id': producto_id, 'nombre': 'Notebook Lenovo'}
```

FastAPI usa los **type hints** de Python (`producto_id: int`) para validar y convertir automáticamente los parámetros — si mandás algo que no es un número, devuelve un `422` sin que escribas ese código vos.

Documentación oficial: [Tutorial de FastAPI](https://fastapi.tiangolo.com/tutorial/)

In [ ]:
!pip install -q fastapi uvicorn

from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get('/')
def inicio():
    return {'mensaje': 'API de productos'}

@app.get('/productos/{producto_id}')
def obtener_producto(producto_id: int):
    return {'id': producto_id, 'nombre': 'Notebook Lenovo'}

cliente = TestClient(app)
resp = cliente.get('/productos/1')
print(resp.status_code, resp.json())

resp = cliente.get('/productos/abc')
print(resp.status_code, resp.json())

### Ejercicio 1 — Endpoint con query params

Agregá a la `app` anterior un endpoint `GET /productos` que reciba dos query params opcionales: `categoria: str = None` y `limite: int = 10`, y devuelva `{'categoria': categoria, 'limite': limite}`. Probalo con `TestClient` pasando `params={'categoria': 'notebooks', 'limite': 5}`.

In [ ]:
# TODO: agregar el endpoint GET /productos con categoria y limite como query params

# TODO: probarlo con cliente.get('/productos', params={...})

<details>
<summary>💡 Ver solución</summary>

```python
from typing import Optional

@app.get('/productos')
def listar_productos(categoria: Optional[str] = None, limite: int = 10):
    return {'categoria': categoria, 'limite': limite}

resp = cliente.get('/productos', params={'categoria': 'notebooks', 'limite': 5})
print(resp.status_code, resp.json())
```

</details>

## 3. Modelos con Pydantic

Pydantic define la forma de tus datos con clases Python normales, y valida automáticamente tipos, campos requeridos/opcionales y restricciones.

```python
from pydantic import BaseModel, Field

class ProductoIn(BaseModel):
    nombre: str
    precio: float = Field(gt=0)
    categoria: str = 'general'
    stock: int = 0
```

Si un cliente manda un precio negativo o se olvida el nombre, FastAPI devuelve automáticamente un `422` con el detalle del error, sin que escribas esa validación a mano.

Documentación oficial: [Pydantic](https://docs.pydantic.dev/latest/) · [Request Body en FastAPI](https://fastapi.tiangolo.com/tutorial/body/)

In [ ]:
from pydantic import BaseModel, Field

class ProductoIn(BaseModel):
    nombre: str
    precio: float = Field(gt=0)
    categoria: str = 'general'
    stock: int = 0

@app.post('/productos', status_code=201)
def crear_producto(producto: ProductoIn):
    return producto

cliente = TestClient(app)

resp = cliente.post('/productos', json={'nombre': 'Mouse', 'precio': 15.5})
print(resp.status_code, resp.json())

resp = cliente.post('/productos', json={'nombre': 'Mouse', 'precio': -5})
print(resp.status_code, resp.json())

### Ejercicio 2 — Validar con Pydantic

Agregá a `ProductoIn` un campo `sku: str` que sea obligatorio y tenga como mínimo 4 caracteres (usá `Field(min_length=4)`). Probá crear un producto sin `sku` y confirmá que devuelve `422`.

<details>
<summary>💡 Ver solución</summary>

```python
class ProductoIn(BaseModel):
    nombre: str
    precio: float = Field(gt=0)
    categoria: str = 'general'
    stock: int = 0
    sku: str = Field(min_length=4)

@app.post('/productos-v2', status_code=201)
def crear_producto_v2(producto: ProductoIn):
    return producto

resp = cliente.post('/productos-v2', json={'nombre': 'Mouse', 'precio': 15.5})
print(resp.status_code, resp.json())  # 422, falta sku
```

</details>

## 4. Serialización y `response_model`

Así como un modelo Pydantic valida lo que entra, otro modelo puede definir la forma exacta de lo que sale — útil para no exponer campos internos (por ejemplo, un costo interno que el cliente no debería ver).

```python
class ProductoOut(BaseModel):
    id: int
    nombre: str
    precio: float

@app.post('/productos-v3', response_model=ProductoOut, status_code=201)
def crear_producto_v3(producto: ProductoIn):
    return {**producto.model_dump(), 'id': 1, 'costo_interno': 8.0}  # costo_interno se descarta
```

FastAPI filtra automáticamente la respuesta según `response_model`, aunque la función devuelva campos de más.

### Ejercicio 3 — `response_model`

Creá un endpoint `GET /productos/{producto_id}/resumen` que devuelva un diccionario con `id`, `nombre`, `precio` y también `margen_secreto` (un dato interno), pero usando `response_model=ProductoOut` para que `margen_secreto` no llegue nunca al cliente. Confirmá el resultado con `TestClient`.

<details>
<summary>💡 Ver solución</summary>

```python
@app.get('/productos/{producto_id}/resumen', response_model=ProductoOut)
def resumen_producto(producto_id: int):
    return {'id': producto_id, 'nombre': 'Mouse', 'precio': 15.5, 'margen_secreto': 40}

resp = cliente.get('/productos/1/resumen')
print(resp.status_code, resp.json())  # sin margen_secreto
```

</details>

## 5. Documentación automática: Swagger UI y OpenAPI

FastAPI genera, sin configuración extra, un esquema **OpenAPI** a partir de tus rutas y modelos Pydantic, y lo expone en dos interfaces interactivas cuando corrés la API de verdad:

- `/docs` → **Swagger UI**: probar cada endpoint desde el navegador.
- `/redoc` → **ReDoc**: documentación de referencia, más orientada a lectura.

Podés enriquecer la documentación con metadata:

```python
app = FastAPI(title='API de Productos', version='1.0.0', description='Microservicio de catalogo')

@app.get('/productos/{producto_id}', summary='Obtener un producto por id', tags=['productos'])
def obtener_producto(producto_id: int):
    ...
```

Documentación oficial: [Metadata y docs en FastAPI](https://fastapi.tiangolo.com/tutorial/metadata/) · [Especificación OpenAPI](https://swagger.io/specification/) · [Swagger UI](https://swagger.io/tools/swagger-ui/)

### Ejercicio 4 — Enriquecer la documentación

Agregále a la `app` un `title`, `version` y `description`, y al endpoint de creación de productos un `summary` y un `tags=['productos']`. (Este ejercicio se valida mirando `/docs` en el navegador cuando corrés la API con `uvicorn`, más adelante en este notebook.)

<details>
<summary>💡 Ver solución</summary>

```python
app = FastAPI(
    title='API de Productos',
    version='1.0.0',
    description='Microservicio de catalogo de productos',
)

@app.post('/productos', status_code=201, summary='Crear un producto', tags=['productos'])
def crear_producto(producto: ProductoIn):
    return producto
```

</details>

## 6. Correr la API de verdad

`TestClient` es perfecto para probar dentro del notebook, pero para ver `/docs` en el navegador necesitás un servidor real:

```bash
uvicorn main:app --reload
```

Esto la deja disponible en `http://127.0.0.1:8000/docs`. En Colab, para exponerla públicamente y poder abrir esa URL, se suele usar un túnel (por ejemplo con [pyngrok](https://pyngrok.readthedocs.io/en/latest/)) — algo opcional y fuera del alcance de este notebook, pero útil si querés mostrar la documentación interactiva en vivo.

## 7. El mismo endpoint en Flask (comparación)

```python
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/productos', methods=['POST'])
def crear_producto():
    data = request.get_json()
    if not data.get('nombre') or data.get('precio', 0) <= 0:
        return jsonify({'error': 'datos invalidos'}), 422
    return jsonify(data), 201
```

En Flask, la validación (el `if` de arriba) y la documentación quedan a tu cargo o requieren extensiones adicionales — es el trade-off frente a la ergonomía automática de FastAPI + Pydantic.

## Mini-proyecto: CRUD de tareas

Construí, en FastAPI, un CRUD completo en memoria (una lista de Python como base de datos) para un recurso `tarea` con: `id`, `titulo`, `completada: bool = False`.

- `POST /tareas` — crear (Pydantic valida que `titulo` no esté vacío)
- `GET /tareas` — listar, con query param opcional `completada`
- `GET /tareas/{id}` — obtener una, `404` si no existe
- `PUT /tareas/{id}` — reemplazar
- `DELETE /tareas/{id}` — borrar

Probá los 5 endpoints con `TestClient`.

**Entregable:** el código de la API + las pruebas con `TestClient` mostrando cada caso (incluido el `404`).

---

**Seguís en:** *Colab 3 — Microservicio de predicciones y métricas de negocio*